# 32 — W6 B3 stage: Rank-GRPO main loop for the responder LLM

Per RecSys_Challenge_Plan §6.3 row B3 + §6.4 + §6.5. **Main training stage** of Component-B: refines the envelope-fluent (W4 KTO + optional W5 S-DPO) responder by gradient-following the composite per-turn reward `compose_r_turn`. Retriever (CMQR + ProRank + catalog filter) is FROZEN at W6 start; only the responder's text generation is on-policy.

## Why Rank-GRPO (plan §6.3)

Once the responder reliably emits the envelope (W4 + optionally W5), Rank-GRPO is what actually improves the response *content* against the leaderboard-anchored reward. We use **TRL `GRPOTrainer`** with `compose_r_turn` plugged in as the `reward_funcs` callback. R_retr is constant per row (frozen retriever) → in standard GRPO, the per-prompt advantage normalization `A_i = (r_i − mean(r)) / std(r)` cancels the constant, so R_retr's 0.70 weight contributes ZERO gradient. Useful gradient mass is **R_rule (0.15) + R_format (0.05) = 0.20** (R_judge is 0 at training time without an external judge).

## Pipeline

1. Mount Drive, install deps, HF auth.
2. Gate-check: resolve starting base = W5 merged > W4 merged. Abort if W4 hasn't merged.
3. Pytest pre-flight (W1–W6 modules).
4. **Retrieval pre-compute (heavy)**: load CRS_BASELINE-style stack (CMQR + ProRank w/ rationales) and run over every unique POS turn → `data/trl/grpo_retrieval.parquet`.
5. Build the GRPO parquet on-the-fly: `build_grpo_dataset.py` joins envelope + retrieval. **Conversational prompt format** (W6 review P0-3 fix).
6. TRL GRPOTrainer on (Qwen-7B + W5/W4-merged base) + fresh LoRA r=32 + `compose_r_turn` reward. **G=4** (W6 review P1-1 fix vs. plan's G=2 to keep the GRPO advantage informative); **scale_rewards=False** so the magnitude of `r_rollout_a − r_rollout_b` survives the std normalization.
7. **In-place merge-and-push** deployment artifact (W6 review P1-5 fix vs. round-trip through Hub).
8. Format-compliance + reward delta + 50-rollout qual review (plan §6.3 row B3 gate). Eval feeds the same conversational prompts the trainer saw.

## Compute

- ~10 A100-hr for ~12k optimizer steps × G=4 rollouts (P1-1: cut max_steps from 25k→12k to compensate for G=2→4).
- Plus ~30 min retrieval pre-compute.
- Plus ~5 min Hub upload of merged 7B (~14 GB).
- **Total: ~12 A100-hr for one W6 pass.**

## Gate (plan §6.3 row B3)

- **Reward delta:** R_turn (W6) − R_turn (B1=W4) ≥ +0.03 on a 50-row eval slice.
- **Format compliance:** strict r_format ≥ 95% (must not regress vs W5).
- **Dev nDCG@20:** not regressed > 0.005 vs frozen-retriever baseline (deferred — checked in W7 integration via `colab/40_run_blindset_B.ipynb`).

## Review fixes applied (researcher review 2026-05-02)

P0-1 cell 4 retrieval pre-compute now uses real `mcrs.crs_baseline.CRS_BASELINE`-style instantiation (CMQR_REWRITER + ProRank with `with_rationales=True`); old code referenced non-existent `mcrs.retrieval.wrrf.WRRFRetriever`. P0-2 dropped `max_prompt_length` (removed from TRL ≥0.25). P0-3 prompts emitted in conversational form via `--system-prompt-path` so the chat template is auto-applied. P0-4 fixed `_HISTORY_RE` to capture multi-line history. P1-1 G=4 + `scale_rewards=False`. P1-2 split format reward into a separate `reward_funcs` entry with `reward_weights` so format failure doesn't multiplicatively zero r_turn at training. P1-3 dedupe added in cell 4 partial-flush. P1-4 normalize message-list completions in reward closure. P1-5 in-place merge via `trainer.model.merge_and_unload()`. P1-6 `report_to='none'` + manual `trackio.log` since trackio isn't a registered HF Trainer integration. P2 LR raised 1e-6→5e-6; peft pinned ≥0.13 for Qwen2.5 `target_modules='all-linear'`.

In [ ]:
# 1) GPU check.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
!git log -1 --pretty=format:'commit:  %h%nsubject: %s'

In [ ]:
# 2b) Mount Drive + persistent caches.
import os, shutil
from google.colab import drive

try: drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}; retrying ...')
    try: drive.flush_and_unmount()
    except Exception: pass
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/grpo_runs']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

In [ ]:
# 3) HF auth — required for push_to_hub.
#
# Setup: Colab → 🔑 Secrets pane → add `HF_TOKEN` with WRITE scope.
# Get the token at https://huggingface.co/settings/tokens.
#
# Fail-fast: aborts immediately if the secret is missing or the token is
# invalid — better than failing 3 hours into training when push_to_hub fires.
import os, sys
from google.colab import userdata
from huggingface_hub import whoami

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise SystemExit(
        f"\u274c HF_TOKEN secret not found in Colab ({e!r}).\n"
        f"   1) Open the \U0001f511 Secrets pane in the left sidebar.\n"
        f"   2) Add a secret named exactly `HF_TOKEN` (case-sensitive).\n"
        f"   3) Toggle 'Notebook access' ON for this notebook."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

try:
    user = whoami(token=HF_TOKEN)
    HF_USERNAME = user["name"]
    os.environ["HF_USERNAME"] = HF_USERNAME
    print(f"\u2713 HF auth ok \u2014 logged in as {HF_USERNAME}")
except Exception as e:
    raise SystemExit(
        f"\u274c HF auth failed: {e!r}\n"
        f"   Token may lack WRITE scope. Regenerate at https://huggingface.co/settings/tokens"
    )

In [ ]:
# 4) Gate-check + starting-base resolution.
#
# W6 always starts from a MERGED base (per project_responder_merge_pattern.md):
#   - If W5 ran, starting base = W5's merged_hub_model.
#   - Else (W4 format ≥70%), starting base = W4's merged_hub_model.
#   - Else: ABORT — W4 must have run + merged before W6.
import json
from pathlib import Path

def latest_gate(runs_dir: str):
    p = Path(runs_dir)
    if not p.exists():
        return None
    candidates = list(p.rglob('gate_result.json'))
    if not candidates:
        return None
    latest = max(candidates, key=lambda x: x.stat().st_mtime)
    with latest.open() as f:
        return json.load(f), latest

W5_RESULT = latest_gate(f'{DRIVE_BASE}/sdpo_runs')
W4_RESULT = latest_gate(f'{DRIVE_BASE}/kto_runs')

STARTING_MERGED = None
PRIOR_STAGE = None
B1_FORMAT = None  # W4 format compliance — used as B1 baseline label.

if W5_RESULT:
    w5, w5_path = W5_RESULT
    if w5.get('merged_hub_model'):
        STARTING_MERGED = w5['merged_hub_model']
        PRIOR_STAGE = 'W5'
        print(f'W5 latest: {w5_path}')
        print(f'  format compliance: {w5.get("format_compliance_strict", 0):.1%}')
        print(f'  merged repo:       {STARTING_MERGED}')

if STARTING_MERGED is None and W4_RESULT:
    w4, w4_path = W4_RESULT
    if w4.get('merged_hub_model'):
        STARTING_MERGED = w4['merged_hub_model']
        PRIOR_STAGE = 'W4'
        print(f'W4 latest: {w4_path}')
        print(f'  format compliance: {w4.get("format_compliance_strict", 0):.1%}')
        print(f'  merged repo:       {STARTING_MERGED}')

if W4_RESULT:
    B1_FORMAT = W4_RESULT[0].get('format_compliance_strict')
B1_REPO = W4_RESULT[0]['merged_hub_model'] if W4_RESULT and W4_RESULT[0].get('merged_hub_model') else None

if STARTING_MERGED is None:
    print('\n❌ No merged base found from W4 or W5.')
    print(f'    Looked under: {DRIVE_BASE}/kto_runs, {DRIVE_BASE}/sdpo_runs')
    print('    Required: at least one gate_result.json with `merged_hub_model` set.')
    raise SystemExit('W6 aborted — no W4/W5 merged base.')

print(f'\n→ starting from {PRIOR_STAGE} merged base: {STARTING_MERGED}')
print(f'  B1 baseline (W4 merged) for reward delta: {B1_REPO or "(missing)"}')

In [ ]:
# 5) Install deps + pytest pre-flight.
# peft>=0.13 required for `target_modules='all-linear'` to resolve correctly
# on Qwen2.5 (older peft silently misses some attention proj names — W6 review P2).
!pip install -q --upgrade transformers datasets 'pandas<3.0' tqdm omegaconf
!pip install -q --upgrade 'trl>=0.12.0' 'peft>=0.13.0' 'torchao>=0.16.0' && pip install -q flash-attn --no-build-isolation || echo 'flash-attn install failed; will fall back to SDPA at model-load time' trackio accelerate
!python -c 'import torch, transformers, trl, peft; print("torch", torch.__version__, "trl", trl.__version__, "peft", peft.__version__)'

# Pytest pre-flight — confirm W1–W6 modules all green.
!cd /content/recsys2026 && python -m pytest \
    tests/test_reward_fns.py \
    tests/test_state_tracker.py \
    tests/test_cmqr.py \
    tests/test_pro_rank.py \
    tests/test_augment_envelope.py \
    tests/test_build_trl_datasets.py \
    tests/test_build_sdpo_dataset.py \
    tests/test_build_grpo_dataset.py \
    -q

In [ ]:
# 6) Retrieval pre-compute (W6-specific HEAVY step).
#
# REWRITTEN per W6 review P0-1 + P2 to use the real mcrs APIs verified in
# the canonical inference path (mcrs.crs_baseline.CRS_BASELINE):
#   - mcrs.retrieval_modules.load_retrieval_module → RRF_MODEL (wRRF).
#   - mcrs.query_rewriters.cmqr.CMQR_REWRITER  (was: 'CMQRRewriter' — DNE).
#   - mcrs.rerankers.pro_rank.ProRankReranker(with_rationales=True) →
#     `rerank()` returns IDs; rationales come from `generate_rationales(query, tids)`.
#   - mcrs.query_rewriters.state_tracker.StateTracker (CMQR requires it).
#   - mcrs.db_item.MusicCatalogDB for top-1 metadata via `id_to_metadata`.
#
# Idempotent: skips if RETRIEVAL_OUT exists. Resumable: flushes every 500 turns.
# Wallclock: ~30 min on A100 for ~15k unique POS turns × top-100 candidates.

import json, sys, re
from pathlib import Path
import pandas as pd
from tqdm import tqdm

ENV_PATH = '/content/recsys2026/data/reward_train_envelope.parquet'
RETRIEVAL_OUT = '/content/recsys2026/data/trl/grpo_retrieval.parquet'
RETRIEVAL_PARTIAL = '/content/recsys2026/data/trl/grpo_retrieval.partial.parquet'

# Build the envelope parquet first if it's not there (chains W1+W4 prep).
N_SESSIONS = 15000
REWARD = '/content/recsys2026/data/reward_train.parquet'
if not Path(REWARD).exists():
    !cd /content/recsys2026 && python scripts/build_reward_dataset.py --n-sessions {N_SESSIONS} --out {REWARD}
if not Path(ENV_PATH).exists():
    !cd /content/recsys2026 && python scripts/augment_envelope.py --in {REWARD} --out {ENV_PATH}

if Path(RETRIEVAL_OUT).exists():
    print(f'reusing existing {RETRIEVAL_OUT}')
else:
    sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
    import torch
    from mcrs.retrieval_modules import load_retrieval_module
    from mcrs.rerankers.pro_rank import ProRankReranker
    from mcrs.query_rewriters.cmqr import CMQR_REWRITER
    from mcrs.query_rewriters.state_tracker import StateTracker
    from mcrs.lm_modules import load_lm_module
    from mcrs.db_item import MusicCatalogDB

    # Match W3 config 110-prorank-rerank-devset.yaml.
    ITEM_DB     = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
    DATASET     = 'talkpl-ai/TalkPlayData-Challenge-Dataset'
    SPLITS      = ['all_tracks']
    CORPUS      = ['track_name', 'artist_name', 'album_name']
    CACHE_DIR   = '/content/recsys2026/music-crs-baselines/experiments/cache'
    LM_TYPE     = 'meta-llama/Llama-3.2-1B-Instruct'   # W2 default for state+CMQR
    PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'

    print('[grpo-retrieval] loading LM (state-tracker + CMQR rewriter)…')
    lm = load_lm_module(
        lm_type=LM_TYPE, device='cuda', attn_implementation='sdpa',
        dtype=torch.bfloat16, use_vllm=False,
    )
    print('[grpo-retrieval] loading wRRF retriever…')
    retrieval = load_retrieval_module(
        retrieval_type='wrrf_bm25_dense_lyrics_v1',
        dataset_name=ITEM_DB, track_split_types=SPLITS,
        corpus_types=CORPUS, cache_dir=CACHE_DIR,
    )
    print('[grpo-retrieval] loading state-tracker…')
    state_tracker = StateTracker(
        lm=lm, prompt_path=f'{PROMPTS_DIR}/state_extraction.txt',
        cache_dir=CACHE_DIR, max_new_tokens=96,
    )
    print('[grpo-retrieval] wrapping with CMQR…')
    cmqr = CMQR_REWRITER(
        lm=lm, inner_retriever=retrieval,
        prompt_path=f'{PROMPTS_DIR}/cmqr_rewrites.txt',
        cache_dir=CACHE_DIR,
        n_rewrites=4, topk_per_rewrite=50, rrf_k=60, max_new_tokens=96,
    )
    print('[grpo-retrieval] loading ProRank reranker (with_rationales=True)…')
    reranker = ProRankReranker(
        item_db_name=ITEM_DB, track_split_types=SPLITS,
        corpus_types=CORPUS, cache_dir=CACHE_DIR,
        with_rationales=True,
    )
    print('[grpo-retrieval] loading item catalog…')
    item_db = MusicCatalogDB(
        dataset_name=ITEM_DB, split_types=SPLITS, corpus_types=CORPUS,
    )
    valid_catalog = set(item_db.metadata_dict.keys())
    print(f'  catalog size: {len(valid_catalog):,}')

    # Filter envelope parquet to POS rows + unique (sid, tn).
    env_df = pd.read_parquet(ENV_PATH).query('label == 1').drop_duplicates(['session_id', 'turn_number'])
    print(f'unique POS turns to retrieve: {len(env_df):,}')

    # Recover gold_track_id from the raw HF dataset (session-nested structure
    # — W6 review P2 catch). Iterate sessions then their `conversations` list.
    from datasets import load_dataset
    print('[grpo-retrieval] loading raw dataset for gold_track_id lookup…')
    raw = load_dataset(DATASET, split='train')
    sid2turn_to_gold: dict = {}
    for sess in raw:
        sid = sess['session_id']
        for msg in sess['conversations']:
            if msg['role'] == 'music':
                sid2turn_to_gold[(sid, int(msg['turn_number']))] = str(msg['content'])

    # Build session_memory by reconstructing prior turns from raw (matching
    # CRS_BASELINE.batch_chat's `session_memory` shape).
    sid2turn_to_history_text = {}
    sid2turn_to_user_query = {}
    for sess in raw:
        sid = sess['session_id']
        # Sort conversations by turn_number then by role priority within turn.
        msgs = sorted(sess['conversations'], key=lambda m: (int(m['turn_number']), m['role']))
        for tn in range(1, 9):
            prior = [m for m in msgs if int(m['turn_number']) < tn]
            history = '\n'.join(f"{m['role']}: {m['content']}" for m in prior)
            user_msg = next((m for m in msgs if int(m['turn_number']) == tn and m['role'] == 'user'), None)
            if user_msg is None:
                continue
            sid2turn_to_user_query[(sid, tn)] = str(user_msg['content'])
            sid2turn_to_history_text[(sid, tn)] = history

    # Resume from partial if it exists. P1-3 fix: dedupe after concat.
    partial_rows = []
    if Path(RETRIEVAL_PARTIAL).exists():
        partial_rows = pd.read_parquet(RETRIEVAL_PARTIAL).to_dict('records')
        partial_df = pd.DataFrame(partial_rows).drop_duplicates(['session_id', 'turn_number'], keep='last')
        partial_rows = partial_df.to_dict('records')
        done_keys = {(r['session_id'], int(r['turn_number'])) for r in partial_rows}
        env_df = env_df[~env_df.apply(lambda r: (r['session_id'], int(r['turn_number'])) in done_keys, axis=1)]
        print(f'resuming — already processed {len(done_keys):,}; remaining {len(env_df):,}')

    rows_out = list(partial_rows)
    flush_every = 500
    BATCH = 16  # batch CMQR + ProRank for throughput

    rows_buffer = []
    for i, row in enumerate(tqdm(env_df.itertuples(index=False), total=len(env_df), desc='retrieve')):
        sid = row.session_id
        tn = int(row.turn_number)
        rows_buffer.append((sid, tn))
        if len(rows_buffer) < BATCH and i + 1 < len(env_df):
            continue

        # ----- run a batch -----
        sids = [s for s, _ in rows_buffer]
        tns  = [t for _, t in rows_buffer]
        queries  = [sid2turn_to_user_query.get((s, t), '') for s, t in rows_buffer]
        histories = [sid2turn_to_history_text.get((s, t), '') for s, t in rows_buffer]

        # State extraction per row.
        states = []
        for s, t, q, h in zip(sids, tns, queries, histories):
            try:
                states.append(state_tracker.extract(s, t, q, h))
            except Exception:
                states.append(None)

        # CMQR retrieval (top-100).
        cmqr.set_batch_context(session_ids=sids, turn_numbers=tns, extracted_states=states)
        try:
            top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100, user_ids=[None] * len(queries))
        except TypeError:
            top100 = cmqr.batch_text_to_item_retrieval(queries, topk=100)

        # ProRank rerank → top-20 IDs.
        top20 = reranker.rerank(queries, top100, topk=20)

        # Catalog filter + dedupe per row (mirrors CRS_BASELINE stage 1c).
        for s, t, q, ids20, pool100 in zip(sids, tns, queries, top20, top100):
            seen, kept = set(), []
            for tid in ids20:
                if tid in seen or tid not in valid_catalog:
                    continue
                kept.append(tid); seen.add(tid)
            if len(kept) < 20:
                for tid in pool100:
                    if len(kept) >= 20:
                        break
                    if tid in seen or tid not in valid_catalog:
                        continue
                    kept.append(tid); seen.add(tid)
            kept = kept[:20]

            # Rationales for the kept tids.
            try:
                rationales = reranker.generate_rationales(q, kept)
            except Exception as e:
                print(f'[grpo-retrieval] rationale gen failed sid={s} tn={t}: {e!r}')
                rationales = ['' for _ in kept]

            # Top-1 metadata.
            top1_tid = kept[0] if kept else ''
            top1_meta = item_db.metadata_dict.get(top1_tid, {}) if top1_tid else {}
            tn_get = lambda field: (top1_meta.get(field) or [''])
            top1_track_name  = (tn_get('track_name')[0] if isinstance(tn_get('track_name'), list) else str(tn_get('track_name'))) or ''
            top1_artist_name = (tn_get('artist_name')[0] if isinstance(tn_get('artist_name'), list) else str(tn_get('artist_name'))) or ''

            rows_out.append({
                'session_id': s,
                'turn_number': t,
                'gold_track_id': sid2turn_to_gold.get((s, t), ''),
                'predicted_track_ids': kept,
                'top1_track_name': top1_track_name,
                'top1_artist_name': top1_artist_name,
                'reranker_rationales': rationales,
            })
        rows_buffer.clear()

        if (i + 1) % flush_every == 0:
            # P1-3 fix: dedupe before flushing partial.
            partial_out = pd.DataFrame(rows_out).drop_duplicates(['session_id', 'turn_number'], keep='last')
            partial_out.to_parquet(RETRIEVAL_PARTIAL, index=False)

    # Final dedupe + write.
    out_df = pd.DataFrame(rows_out).drop_duplicates(['session_id', 'turn_number'], keep='last')
    Path(RETRIEVAL_OUT).parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(RETRIEVAL_OUT, index=False)
    if Path(RETRIEVAL_PARTIAL).exists():
        Path(RETRIEVAL_PARTIAL).unlink()
    print(f'\n✓ retrieval cache: {len(out_df):,} rows → {RETRIEVAL_OUT}')

# Quick sanity print regardless of which path we took.
ret_df = pd.read_parquet(RETRIEVAL_OUT)
print(f'retrieval rows: {len(ret_df):,}')
print(f'unique sessions: {ret_df["session_id"].nunique():,}')
print(f'mean predicted_track_ids len: {ret_df["predicted_track_ids"].apply(len).mean():.1f}')

In [ ]:
# 7) Build the GRPO parquet by joining envelope + retrieval.
# W6 review P0-3 fix: pass --system-prompt-path so prompts are emitted in
# conversational form and TRL auto-applies the chat template at training time.
GRPO_OUT = '/content/recsys2026/data/trl/grpo.parquet'
SYSTEM_PROMPT_TXT = '/content/recsys2026/data/trl/grpo_system_prompt.txt'

# Compose the system prompt the same way crs_baseline.CRS_BASELINE._get_system_prompt
# does (roleplay + response_generation_cot_user_state). Keeping it as a plain
# .txt file lets us pass --system-prompt-path to the build script.
PROMPTS_DIR = '/content/recsys2026/music-crs-baselines/mcrs/system_prompts'
with open(f'{PROMPTS_DIR}/roleplay.txt', encoding='utf-8') as f:
    role_play = f.read()
with open(f'{PROMPTS_DIR}/response_generation_cot_user_state.txt', encoding='utf-8') as f:
    cot_prompt = f.read()
SYSTEM_PROMPT_STR = role_play + '\n\n' + cot_prompt
Path(SYSTEM_PROMPT_TXT).parent.mkdir(parents=True, exist_ok=True)
with open(SYSTEM_PROMPT_TXT, 'w', encoding='utf-8') as f:
    f.write(SYSTEM_PROMPT_STR)
print(f'✓ system prompt written ({len(SYSTEM_PROMPT_STR):,} chars) → {SYSTEM_PROMPT_TXT}')

if not Path(GRPO_OUT).exists():
    !cd /content/recsys2026 && python scripts/build_grpo_dataset.py \
        --envelope {ENV_PATH} \
        --retrieval {RETRIEVAL_OUT} \
        --system-prompt-path {SYSTEM_PROMPT_TXT} \
        --out {GRPO_OUT}
else:
    print(f'reusing existing {GRPO_OUT}')

import pandas as pd
d = pd.read_parquet(GRPO_OUT)
print(f'\nGRPO dataset: {len(d):,} rows')
print(f'columns: {list(d.columns)}')
print(f'splits: {d["split"].value_counts().to_dict()}')

# Verify prompt is conversational (list-of-dicts, not flat string).
sample_prompt = d['prompt'].iloc[0]
import numpy as np
if isinstance(sample_prompt, np.ndarray):
    sample_prompt = list(sample_prompt)
assert isinstance(sample_prompt, list), f'expected conversational prompt, got {type(sample_prompt)}'
assert sample_prompt[0]['role'] == 'system' and sample_prompt[1]['role'] == 'user'
print(f'✓ prompts are conversational ([system, user]) — TRL will auto-apply chat template')

# Abort if join coverage was bad.
n_pos_total = pd.read_parquet(ENV_PATH).query('label == 1').shape[0]
unjoined_frac = (n_pos_total - len(d)) / max(n_pos_total, 1)
print(f'POS coverage: {len(d):,} / {n_pos_total:,} ({100*(1-unjoined_frac):.0f}%)')
if unjoined_frac > 0.30:
    raise SystemExit(f'❌ {100*unjoined_frac:.0f}% of POS turns missing from retrieval; rerun cell 6.')

In [ ]:
# 8) Schema validation + train/val split.
from datasets import Dataset, Features, Value, Sequence
import numpy as np

# parquet roundtrip wraps the message-list as a numpy array; convert back to
# plain Python list-of-dicts so Dataset.from_pandas + TRL handle it cleanly.
def _normalize_prompt(p):
    if isinstance(p, np.ndarray):
        return [dict(m) for m in p]
    if isinstance(p, list) and p and isinstance(p[0], np.ndarray):
        return [dict(m) for m in p]
    return p
d['prompt'] = d['prompt'].apply(_normalize_prompt)

ds = Dataset.from_pandas(d, preserve_index=False)

required = {'prompt', 'gold_track_id', 'predicted_track_ids',
            'top1_meta_json', 'user_state_json', 'history_text', 'split'}
assert required.issubset(set(ds.column_names)), \
    f'missing required columns: {required - set(ds.column_names)}'
print(f'✓ schema PASS — columns: {ds.column_names}')

split = ds.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'train: {len(train_ds):,}  eval: {len(eval_ds):,}')

# Sanity: confirm conversational shape survived Dataset.from_pandas.
sample = train_ds[0]['prompt']
assert isinstance(sample, list) and sample[0]['role'] == 'system'
print(f'✓ conversational prompt shape preserved through Dataset')

In [ ]:
# 9) Trackio init — `report_to='trackio'` is NOT a registered HF Trainer
# integration (W6 review P1-6 fix). We init Trackio directly and log via
# `trackio.log` from our own callbacks; the trainer reports to 'none'.
from datetime import date
import trackio

RUN_NAME = f'b3-grpo-qwen7b-{date.today().isoformat()}'
TRACKIO_OK = True
try:
    trackio.init(
        project='recsys2026',
        name=RUN_NAME,
        # group='b-stage' — removed (current trackio doesn't accept group kwarg; set via config below if desired)
        config={
            'model': STARTING_MERGED,
            'method': 'Rank-GRPO (TRL GRPOTrainer + compose_r_turn reward; on-policy G=4)',
            'prior_stage': PRIOR_STAGE,
            'b1_format': B1_FORMAT,
            'lora_r': 32, 'lora_alpha': 32,
            'num_generations': 4,            # P1-1 fix from G=2
            'scale_rewards': False,          # P1-1 fix
            'kl_beta': 0.04,
            'learning_rate': 5e-6,           # P2 fix from 1e-6
            'max_steps': 12_000,             # P1-1 budget compensation for G=2→4
        },
    )
    print(f'✓ Trackio run: {RUN_NAME}')
except Exception as e:
    TRACKIO_OK = False
    print(f'⚠️  Trackio init failed ({e!r}); training will use console logging only.')


# Custom HF TrainerCallback that pipes logs to Trackio (replaces report_to='trackio').
from transformers import TrainerCallback

class TrackioCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not TRACKIO_OK or not logs:
            return
        try:
            trackio.log({k: float(v) for k, v in logs.items() if isinstance(v, (int, float))})
        except Exception as e:
            print(f'[trackio] log failed: {e!r}')

In [ ]:
# 10) Rank-GRPO training — Qwen-7B + W5/W4-merged base + fresh LoRA r=32.
#
# Deep-review P0-1 fix: this cell now wires `DistilledJudge` (Option B
# refactor). Without this, R_judge contributed 0 gradient at training and
# 30% of the reward weight was inert.
#
# JUDGE_HUB_REPO resolution: prefer an explicit hardcode (after running
# colab/31p), else read from the latest judge_runs/ gate. If neither
# exists, judge falls back to stub (returns 0.0) — RUN AT YOUR OWN RISK.
import json as _json
from pathlib import Path as _P
import os as _os

JUDGE_HUB_REPO = _os.environ.get('JUDGE_HUB_REPO', None)
if JUDGE_HUB_REPO is None:
    # Fallback: try the latest pilot run's gate_result.json.
    pilot_runs = _P(f'{DRIVE_BASE}/grpo_pilot_runs')
    candidates = list(pilot_runs.rglob('gate_result.json')) if pilot_runs.exists() else []
    if candidates:
        latest = max(candidates, key=lambda x: x.stat().st_mtime)
        with latest.open() as _f:
            _gate = _json.load(_f)
        JUDGE_HUB_REPO = _gate.get('judge')

if JUDGE_HUB_REPO is None:
    print('⚠️  No JUDGE_HUB_REPO set and no pilot gate found. R_judge will be 0.0.')
    print('   Set env JUDGE_HUB_REPO=<repo> or run colab/31p first to train + record one.')
else:
    print(f'✓ Using distilled judge: {JUDGE_HUB_REPO}')

import torch, gc, json, sys
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import compose_r_turn, r_format, DistilledJudge

# Instantiate + warm up the judge before trainer init (P1-4 fix).
JUDGE = DistilledJudge(checkpoint=JUDGE_HUB_REPO)
JUDGE.warmup()  # forces Hub download + first forward NOW, not on step 1

HUB_REPO = f'{HF_USERNAME}/recsys2026-{RUN_NAME}'
OUTPUT_DIR = f'/content/recsys2026/training_runs/{RUN_NAME}'

peft_config = LoraConfig(
    r=32, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM', target_modules='all-linear',
)


def _completion_text(completion):
    if isinstance(completion, list) and completion:
        last = completion[-1]
        if isinstance(last, dict):
            return str(last.get('content', ''))
        return str(last)
    return str(completion)


def _user_content_from_prompt(prompt):
    """Extract the user-role content for the judge's `context` arg."""
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get('role') == 'user':
                return str(msg.get('content', ''))
        return ''
    return str(prompt)


def reward_main(prompts, completions, **kwargs):
    """Composite Option B reward (deep-review P0-1: judge + group bonus wired).
    Format gate stays in the separate reward_format fn so a bad rollout
    doesn't multiplicatively zero the gradient.
    """
    from collections import defaultdict
    groups = defaultdict(list)
    sids = kwargs['session_id']; tns = kwargs['turn_number']
    for i in range(len(completions)):
        groups[(sids[i], tns[i])].append(i)

    contexts = [_user_content_from_prompt(p) for p in prompts]
    completion_texts = [_completion_text(c) for c in completions]
    judge_scores = JUDGE.score_batch(contexts, completion_texts, batch_size=16)

    scores = []
    for i, completion in enumerate(completions):
        peer_indices = groups[(sids[i], tns[i])]
        peer_responses = [completion_texts[j] for j in peer_indices]
        comps = compose_r_turn(
            predicted_track_ids=list(kwargs['predicted_track_ids'][i]),
            gold_track_id=kwargs['gold_track_id'][i],
            response_text=completion_texts[i],
            valid_catalog=None,
            top1_meta=json.loads(kwargs['top1_meta_json'][i]),
            user_state=json.loads(kwargs['user_state_json'][i]),
            user_profile=json.loads(kwargs['user_profile_json'][i]) if 'user_profile_json' in kwargs else None,  # gap-analysis Step 3: user_profile piped
            history_text=kwargs['history_text'][i],
            judge_score=judge_scores[i],
            include_format=False,
            group_responses=peer_responses,
        )
        scores.append(comps['r_turn'])
    return scores


def reward_format(prompts, completions, **kwargs):
    return [r_format(_completion_text(c)) for c in completions]


config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    model_init_kwargs={"torch_dtype": "bfloat16", "attn_implementation": "flash_attention_2"},  # FA2 ~30% speedup; if unavailable TRL falls back automatically
    push_to_hub=True, hub_model_id=HUB_REPO, hub_strategy='every_save', hub_private_repo=True,
    num_generations=4,
    scale_rewards=False,
    max_completion_length=256,  # was 320 → 256 (~20% rollout speedup)
    temperature=0.9, beta=0.04,
    reward_weights=[0.95, 0.05],
    max_steps=12_000,
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    learning_rate=5e-6, lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=True, gradient_checkpointing=True,
    eval_strategy='no',  # speedup: skip mid-training eval eval_steps=500, per_device_eval_batch_size=2,
    save_strategy='steps', save_steps=250, save_total_limit=3,
    logging_steps=20, report_to='none',
)

callbacks = [TrackioCallback()] if TRACKIO_OK else []

trainer = GRPOTrainer(
    model=STARTING_MERGED, args=config,
    train_dataset=train_ds, eval_dataset=eval_ds,
    reward_funcs=[reward_main, reward_format],
    peft_config=peft_config, callbacks=callbacks,
)

print(f'🚀 Rank-GRPO with DistilledJudge (~10 A100-hr)...')
print(f'   model:   {STARTING_MERGED}  ({PRIOR_STAGE}-merged base)')
print(f'   judge:   {JUDGE_HUB_REPO or "STUB (R_judge=0)"}')
print(f'   adapter → {HUB_REPO}')
print(f'   G={config.num_generations}  scale_rewards={config.scale_rewards}  '
      f'effective_bs={config.per_device_train_batch_size * config.gradient_accumulation_steps} × G')
trainer.train()
print('✓ training complete')

In [ ]:
# 11) Push adapter to Hub + Drive backup.
trainer.push_to_hub()
print(f'✓ adapter at https://huggingface.co/{HUB_REPO}')

import shutil
drive_dst = f'{DRIVE_BASE}/grpo_runs/{RUN_NAME}'
shutil.copytree(OUTPUT_DIR, drive_dst, dirs_exist_ok=True)
print(f'✓ adapter mirrored to {drive_dst}')

In [ ]:
# 11b) DEPLOYMENT MERGE — IN-PLACE (W6 review P1-5 fix).
#
# Old approach: free trainer → re-download STARTING_MERGED from Hub at bf16
# (~14 GB) → re-load W6 LoRA → merge → push. Burns 5-10 min and 14 GB of
# bandwidth.
#
# New approach: trainer.model already has the W6 LoRA wrapping STARTING_MERGED
# in VRAM. Free the optimizer state (no longer needed) and call
# `merge_and_unload()` directly. Saves the Hub round-trip.
#
# Production config 220 references the merged repo with `lora_path: null`.
import gc
import torch
from transformers import AutoTokenizer

# Free optimizer state — biggest VRAM consumer after model weights.
trainer.optimizer = None
trainer.lr_scheduler = None
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f'free VRAM after optimizer del: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB')

# Merge the W6 LoRA into the (already-merged) base in-place.
print(f'merging W6 LoRA into {STARTING_MERGED} (in-place)…')
fully_merged = trainer.model.merge_and_unload()
print(f'  ✓ fully-merged 7B ready (was {PRIOR_STAGE}-merged + W6 LoRA)')

# Tokenizer comes from the starting base.
tok = AutoTokenizer.from_pretrained(STARTING_MERGED)

MERGED_REPO = f'{HF_USERNAME}/recsys2026-{RUN_NAME}-merged'
print(f'pushing fully-merged 7B → {MERGED_REPO} (~3-5 min)...')
fully_merged.push_to_hub(MERGED_REPO, private=True,
                         commit_message=f'W6 Rank-GRPO merged on {STARTING_MERGED}')
tok.push_to_hub(MERGED_REPO, private=True)
print(f'✓ deployment artifact at https://huggingface.co/{MERGED_REPO}')
print(f'  → set lm_type in config/220 to this repo, leave lora_path: null')

In [ ]:
# 12) Format-compliance + reward delta + 50-rollout qual review.
#
# Deep-review P1-7 fix: gate eval now uses the SAME DistilledJudge that
# trained the policy. Without this, the gate measured pre-Option-B reward
# while training optimized post-Option-B reward — non-comparable.
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

W6_MODEL_NAME = MERGED_REPO
print(f'loading W6 merged model: {W6_MODEL_NAME}')
w6_tok = AutoTokenizer.from_pretrained(W6_MODEL_NAME)
w6_model = AutoModelForCausalLM.from_pretrained(
    W6_MODEL_NAME, torch_dtype=torch.bfloat16, device_map='cuda',
).eval()

if not B1_REPO:
    print('⚠️  No W4 merged repo found — skipping B1 reward delta.')
    b1_model = None
else:
    print(f'loading B1 (W4) merged model: {B1_REPO}')
    b1_model = AutoModelForCausalLM.from_pretrained(
        B1_REPO, torch_dtype=torch.bfloat16, device_map='cuda',
    ).eval()

import sys
sys.path.insert(0, '/content/recsys2026/scripts')
from reward_fns import r_format, ENVELOPE, compose_r_turn

def gen(model, tok, conversational_prompt) -> str:
    formatted = tok.apply_chat_template(
        conversational_prompt, tokenize=False, add_generation_prompt=True,
    )
    enc = tok(formatted, return_tensors='pt', truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out_ids = model.generate(
            **enc, max_new_tokens=320, do_sample=False,
            pad_token_id=tok.pad_token_id or tok.eos_token_id,
        )
    return tok.decode(out_ids[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

n_eval = min(50, len(eval_ds))
n_strict = n_loose = 0
samples = []
w6_rewards = []
b1_rewards = []
for i in range(n_eval):
    ex = eval_ds[i]
    conv = ex['prompt']

    w6_out = gen(w6_model, w6_tok, conv)
    if r_format(w6_out) == 1.0: n_strict += 1
    if ENVELOPE.search(w6_out): n_loose += 1
    if len(samples) < 3: samples.append(w6_out[:400])

    user_ctx = conv[1]['content'] if isinstance(conv, list) and len(conv) > 1 else ''
    top1_meta = json.loads(ex['top1_meta_json'])
    user_state = json.loads(ex['user_state_json'])

    w6_comps = compose_r_turn(
        predicted_track_ids=list(ex['predicted_track_ids']),
        gold_track_id=ex['gold_track_id'],
        response_text=w6_out, valid_catalog=None,
        top1_meta=top1_meta, user_state=user_state,
        history_text=ex['history_text'],
        judge_score=JUDGE.score(user_ctx, w6_out),  # P1-7: judge in gate
    )
    w6_rewards.append(w6_comps['r_turn'])

    if b1_model is not None:
        b1_out = gen(b1_model, w6_tok, conv)
        b1_comps = compose_r_turn(
            predicted_track_ids=list(ex['predicted_track_ids']),
            gold_track_id=ex['gold_track_id'],
            response_text=b1_out, valid_catalog=None,
            top1_meta=top1_meta, user_state=user_state,
            history_text=ex['history_text'],
            judge_score=JUDGE.score(user_ctx, b1_out),
        )
        b1_rewards.append(b1_comps['r_turn'])

compliance_strict = n_strict / n_eval
compliance_loose = n_loose / n_eval
mean_w6_r = sum(w6_rewards) / max(len(w6_rewards), 1)
mean_b1_r = sum(b1_rewards) / max(len(b1_rewards), 1) if b1_rewards else None
delta = (mean_w6_r - mean_b1_r) if mean_b1_r is not None else None

print(f'\nFORMAT COMPLIANCE (post-W6):')
print(f'  strict r_format:  {n_strict}/{n_eval} = {compliance_strict:.1%}')
print(f'  loose ENVELOPE:   {n_loose}/{n_eval} = {compliance_loose:.1%}')
print(f'\nREWARD DELTA (with DistilledJudge):')
print(f'  W6 mean R_turn:   {mean_w6_r:.4f}')
if mean_b1_r is not None:
    print(f'  B1 mean R_turn:   {mean_b1_r:.4f}')
    print(f'  Δ vs B1:          {delta:+.4f}  (gate: ≥ +0.0300)')

print('\nSample generations:')
for i, s in enumerate(samples, 1):
    print(f'\n--- sample {i} ---\n{s}')

print('\n' + '=' * 60)
print('W6 B3 GATE (plan §6.3 row B3):')
gate_format = compliance_strict >= 0.95
gate_reward = (delta is not None and delta >= 0.03)
if gate_format and gate_reward:
    print(f'  PASS  format {compliance_strict:.1%}  Δ R_turn {delta:+.4f}')
elif gate_format and delta is None:
    print(f'  PARTIAL — format OK; reward delta not measurable.')
else:
    fails = []
    if not gate_format: fails.append(f'format {compliance_strict:.1%} < 95%')
    if delta is not None and not gate_reward: fails.append(f'Δ R_turn {delta:+.4f} < +0.03')
    print('  FAIL:  ' + '; '.join(fails))
print('=' * 60)

if TRACKIO_OK:
    trackio.log({
        'format_compliance_strict': compliance_strict,
        'format_compliance_loose': compliance_loose,
        'mean_w6_r_turn': mean_w6_r,
        'mean_b1_r_turn': mean_b1_r if mean_b1_r is not None else 0.0,
        'delta_r_turn_vs_b1': delta if delta is not None else 0.0,
    })

In [ ]:
# 13) Persist gate result + finish Trackio.
import json, os
from datetime import date
result = {
    'stage': 'B3-Rank-GRPO',
    'run_name': RUN_NAME,
    'date': date.today().isoformat(),
    'hub_model': HUB_REPO,
    'merged_hub_model': MERGED_REPO,
    'starting_base': STARTING_MERGED,
    'prior_stage': PRIOR_STAGE,
    'b1_format': B1_FORMAT,
    'format_compliance_strict': compliance_strict,
    'format_compliance_loose': compliance_loose,
    'mean_w6_r_turn': mean_w6_r,
    'mean_b1_r_turn': mean_b1_r,
    'delta_r_turn_vs_b1': delta,
    'gate_format_passed': compliance_strict >= 0.95,
    'gate_reward_passed': (delta is not None and delta >= 0.03),
    'gate_passed': (compliance_strict >= 0.95) and (delta is not None and delta >= 0.03),
    'n_eval_samples': n_eval,
    'sample_outputs': samples,
    'config': {
        'method': 'Rank-GRPO (TRL GRPOTrainer + compose_r_turn + split format reward)',
        'num_generations': 4,
        'scale_rewards': False,
        'lora_r': 32,
        'kl_beta': 0.04,
        'lr': 5e-6,
        'max_steps': 12_000,
        'reward_weights': [0.95, 0.05],
    },
    'review_fixes_applied': [
        'P0-1 cell-7 mcrs API rewrite',
        'P0-2 dropped max_prompt_length',
        'P0-3 conversational prompt format',
        'P0-4 multi-line history regex',
        'P1-1 G=4 + scale_rewards=False',
        'P1-2 split format reward',
        'P1-3 dedupe in cell-7 partial flush',
        'P1-4 normalize message-list completions',
        'P1-5 in-place merge_and_unload',
        'P1-6 report_to=none + manual trackio callback',
        'P2 lr 1e-6→5e-6, peft>=0.13',
    ],
}
out_path = f'{DRIVE_BASE}/grpo_runs/{RUN_NAME}/gate_result.json'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f'gate result → {out_path}')

if TRACKIO_OK:
    trackio.finish()
print('✓ done')

## After the run

**B3 PASSES (`format ≥ 95%` AND `Δ R_turn ≥ +0.03`):**
- Adapter at `https://huggingface.co/{HUB_REPO}` (private), merged repo at `{MERGED_REPO}`.
- Run `colab/40_run_blindset_B.ipynb` (TBD) with `lm_type=MERGED_REPO`, `lora_path=null` to verify dev nDCG@20 not regressed > 0.005.
- Proceed to W7 (integration + retrain on train+dev → first Blind-B submission).

**B3 PARTIAL (format passes, reward delta missing or below 0.03):**
- Likely: rewards are noisy. Increase `n_eval` to 200+ rows for a tighter delta estimate.
- Or: `kl_beta 0.04 → 0.02` to give the policy more room to deviate from the W5/W4 init.

**B3 FAILS (format regression OR reward delta clearly negative):**
- Format regression: KL was too loose; raise `kl_beta` and re-run.
- Reward delta negative: monitor `frac_reward_zero_std` from cell 11. If >50% of steps had no advantage signal (both rollouts mis-format), the split-format reward (P1-2) wasn't enough — try raising `reward_weights=[0.90, 0.10]` to put more weight on the format term, or warm-start with a few hundred SFT-style steps before kicking in GRPO.

## Cost-saving knobs

1. **Reduce `max_steps` 12k → 6k** — ~5 A100-hr instead of 10. Loses convergence quality but format-fluency carries from W4/W5.
2. **`G=4 → G=2`** — undoes the P1-1 fix; only do this if VRAM forces it. Pair with `scale_rewards=False` to keep magnitude info.
3. **Reduce `max_completion_length` 320 → 160** — halves rollout time. Risk: truncated answers learned.
4. **`lora_r 32 → 16`** — ~50% fewer trainable params. Plan §6.3.1 mandates 32; document the deviation if used.